# Feature Extraction & Exploration
## Dysgraphia Handwriting Analysis (Computer Vision & Graphology Features)

Notebook ini berfokus pada **Ekstraksi Fitur (*Feature Extraction*)** dan **Eksplorasi Fitur (*Feature EDA*)** dari citra tulisan tangan untuk mendeteksi potensi disgrafia.

Dalam analisis disgrafia medis dan grafologi, karakteristik tulisan tangan anak dengan potensi disgrafia memiliki indikator visual khas:
1. **Ketidakstabilan Garis Dasar (*Baseline Instability*)**: Tulisan naik-turun atau bergelombang (*waviness*).
2. **Inkonsistensi Ukuran Huruf (*Size Variation*)**: Huruf kadang terlalu besar, kadang kecil dalam satu baris kata.
3. **Inkonsistensi Kemiringan (*Slant Angle Variability*)**: Sudut kemiringan goresan berubah-ubah secara acak.
4. **Variabilitas Spasi (*Spacing Irregularity*)**: Jarak antar karakter/kata tidak beraturan.
5. **Kekasaran Goresan & Tekanan (*Stroke Roughness & Pressure*)**: Goresan gemetar (*tremor*), garis putus-putus, atau ketebalan goresan tidak stabil.
6. **Karakteristik Tekstur (*GLCM Texture Features*)**: Pola sebaran piksel tinta dan latar belakang.

---

### 📌 Struktur Notebook
1. **Import Libraries & Konfigurasi**
2. **Pipeline Preprocessing Citra** (Binarization Otsu, Noise Removal, Tight Crop)
3. **Visualisasi Tahapan Preprocessing** (Membandingkan Citra Asli vs Hasil Binarisasi)
4. **Implementasi Ekstraksi Fitur Disgrafia** (Spasial, Geometri, Slant, Spacing, Stroke, GLCM)
5. **Ekstraksi Seluruh Dataset ke DataFrame**
6. **Analisis Statistik & Uji Signifikansi Fitur** (Mann-Whitney U Test / p-value)
7. **Visualisasi Fitur Paling Diskriminatif** (Boxplot & Violin Plot)
8. **Analisis Korelasi Antar Fitur** (Correlation Matrix Heatmap)
9. **Reduksi Dimensi & Visualisasi Sebaran** (PCA & t-SNE 2D Plot)
10. **Peringkat Kepentingan Fitur (*Feature Importance Ranking*)**
11. **Ekspor Dataset Fitur ke CSV (`dysgraphia_features.csv`)**


---
## 1. Import Libraries & Konfigurasi
Memuat library computer vision (`cv2`, `skimage`), statistik numerik (`scipy`, `numpy`, `pandas`), serta visualisasi (`matplotlib`, `seaborn`).


In [ ]:
import os
import math
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Library Feature Extraction & Image Processing
from scipy import stats
from scipy.ndimage import distance_transform_edt
from skimage.feature import graycomatrix, graycoprops
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

try:
    from IPython.display import display
except ImportError:
    display = print

# Konfigurasi gaya visualisasi
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 100

# Set Random Seed
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print("[INFO] Seluruh library pemrosesan citra dan ekstraksi fitur berhasil dimuat.")


---
## 2. Konfigurasi Path Dataset
Menyiapkan direktori dataset gambar tulisan tangan.


In [ ]:
DATASET_DIR = "."

dataset_path = Path(DATASET_DIR).resolve()
print(f"[INFO] Path Dataset: {dataset_path}")

classes = [d.name for d in dataset_path.iterdir() if d.is_dir() and not d.name.startswith('.')]
print(f"[INFO] Kelas ditemukan ({len(classes)}): {classes}")


---
## 3. Pipeline Preprocessing Citra Tulisan Tangan
Sebelum ekstraksi fitur dilakukan, setiap citra melalui tahap pra-pemrosesan standar:
1. **Grayscale Conversion**: Memastikan citra dalam format 1 kanal (intensitas 0-255).
2. **Otsu Binarization (Inverted)**: Menghasilkan citra biner di mana piksel tinta bernilai `255` (foreground/putih) dan latar kertas bernilai `0` (background/hitam).
3. **Morphological Noise Removal**: Membersihkan bintik noise kecil menggunakan operasi *opening*.
4. **Tight Bounding Box Cropping**: Memotong margin putih kosong di sekitar tulisan agar perhitungan spasial murni terfokus pada area tulisan.


In [ ]:
def preprocess_handwriting_image(img_path):
    """
    Melakukan preprocessing pada citra tulisan tangan:
    - Binarization (Otsu Thresholding)
    - Noise Filtering
    - Tight Bounding Box Cropping
    
    Returns:
        gray_img (np.ndarray): Citra grayscale terpotong
        binary_img (np.ndarray): Citra biner (tinta = 255, latar = 0) terpotong
    """
    # 1. Buka citra sebagai grayscale
    img = Image.open(img_path).convert('L')
    gray = np.array(img)
    
    # 2. Otsu's Thresholding (Inverted: tinta = 255, kertas = 0)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    # 3. Noise removal kecil (opening)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    binary_clean = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    
    # 4. Tight bounding box cropping (buang whitespace kosong di tepi)
    coords = cv2.findNonZero(binary_clean)
    if coords is not None:
        x, y, w, h = cv2.boundingRect(coords)
        # Beri padding kecil 4px
        pad = 4
        x1 = max(0, x - pad)
        y1 = max(0, y - pad)
        x2 = min(gray.shape[1], x + w + pad)
        y2 = min(gray.shape[0], y + h + pad)
        
        cropped_gray = gray[y1:y2, x1:x2]
        cropped_binary = binary_clean[y1:y2, x1:x2]
    else:
        cropped_gray = gray
        cropped_binary = binary_clean
        
    return cropped_gray, cropped_binary

print("[INFO] Fungsi preprocessing berhasil didefinisikan.")


---
## 4. Visualisasi Tahapan Preprocessing
Menampilkan perbandingan antara citra asli dan hasil binarisasi terpotong (*cropped binary*) untuk memverifikasi bahwa garis tulisan terekstraksi dengan bersih tanpa kehilangan komponen huruf.


In [ ]:
def visualize_preprocessing_samples(base_dir, n_samples=3):
    """
    Menampilkan perbandingan citra asli vs hasil preprocessing dari kedua kelas.
    """
    base_path = Path(base_dir).resolve()
    classes = [d for d in base_path.iterdir() if d.is_dir() and not d.name.startswith('.')]
    
    fig, axes = plt.subplots(len(classes) * n_samples, 2, figsize=(12, len(classes) * n_samples * 2.2))
    row = 0
    
    for cls in classes:
        files = list(cls.glob("*.jpg")) + list(cls.glob("*.png"))
        sampled_files = random.sample(files, min(n_samples, len(files)))
        
        for f in sampled_files:
            orig = Image.open(f)
            gray, binary = preprocess_handwriting_image(f)
            
            axes[row, 0].imshow(orig, cmap='gray')
            axes[row, 0].set_title(f"Original: {cls.name}\n{f.name} ({orig.size[0]}x{orig.size[1]})", fontsize=9)
            axes[row, 0].axis('off')
            
            axes[row, 1].imshow(binary, cmap='gray')
            axes[row, 1].set_title(f"Binarized & Cropped\n({binary.shape[1]}x{binary.shape[0]})", fontsize=9)
            axes[row, 1].axis('off')
            
            row += 1
            
    plt.suptitle("Perbandingan Citra Asli vs Hasil Preprocessing Binarisasi", fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

visualize_preprocessing_samples(DATASET_DIR, n_samples=2)


---
## 5. Implementasi Ekstraksi Fitur Disgrafia
Ekstraksi fitur kuantitatif mencakup 6 kelompok karakteristik tulisan tangan:

### A. Fitur Garis Dasar (*Baseline Stability*)
- **`baseline_dev_std`**: Standar deviasi posisi vertikal (koordinat Y) centroid tiap komponen huruf.
- **`baseline_waviness`**: Rata-rata selisih vertikal berturutan antar centroid ($\Delta Y$). Mengukur tingkat gelombang baris tulisan.

### B. Fitur Variasi Ukuran Huruf (*Letter Dimension Consistency*)
- **`height_mean`** & **`height_std`**: Rata-rata dan standar deviasi tinggi komponen huruf.
- **`height_cov`**: *Coefficient of Variation* tinggi huruf ($rac{	ext{height\_std}}{	ext{height\_mean}}$). Indikator utama ketidakkonsistenan ukuran tulisan.
- **`width_mean`**, **`width_std`**, **`width_cov`**: Rata-rata, standar deviasi, dan CoV lebar komponen huruf.
- **`aspect_ratio_char_std`**: Variabilitas proporsi lebar/tinggi karakter.

### C. Fitur Kemiringan (*Slant Angle Variability*)
- **`slant_angle_mean`**: Rata-rata sudut kemiringan goresan vertikal (dihitung dari gradien Sobel).
- **`slant_angle_std`**: Standar deviasi sudut kemiringan (variasi tinggi = kemiringan huruf acak).

### D. Fitur Spasi & Kerapatan Tinta (*Spacing & Ink Density*)
- **`ink_density`**: Rasio piksel tinta terhadap luas bounding box tulisan.
- **`inter_comp_spacing_mean`** & **`inter_comp_spacing_std`**: Rata-rata dan variasi jarak kosong horizontal antar komponen huruf berturutan.
- **`num_components_density`**: Jumlah komponen terhubung per 1000 piksel lebar.

### E. Fitur Kekasaran Goresan (*Stroke Roughness & Tremor*)
- **`stroke_roughness_mean`**: Rasio isoperimetrik $rac{P^2}{4\pi A}$ dari kontur huruf (mengukur ketidakteraturan/kerutan goresan pena).
- **`stroke_thickness_mean`** & **`stroke_thickness_std`**: Estimasi rata-rata dan kestabilan ketebalan goresan menggunakan *Distance Transform*.

### F. Fitur Tekstur GLCM (*Gray-Level Co-occurrence Matrix*)
- **`glcm_contrast`**, **`glcm_dissimilarity`**, **`glcm_homogeneity`**, **`glcm_energy`**, **`glcm_correlation`**.


In [ ]:
def extract_handwriting_features(img_path):
    """
    Mengekstrak kumpulan fitur morfologi, spasial, kontur, dan tekstur dari citra tulisan tangan.
    """
    gray, binary = preprocess_handwriting_image(img_path)
    h_img, w_img = binary.shape
    
    # 1. Deteksi Komponen Terhubung (Connected Components)
    # Area biner: tinta = 255
    num_labels, labels, stats_cc, centroids = cv2.connectedComponentsWithStats(binary, connectivity=8)
    
    # Filter komponen kecil (noise / titik sangat kecil)
    min_area = 15
    valid_indices = [i for i in range(1, num_labels) if stats_cc[i, cv2.CC_STAT_AREA] >= min_area]
    
    if len(valid_indices) < 2:
        # Fallback jika komponen terlalu sedikit
        valid_indices = list(range(1, max(2, num_labels)))
        
    comp_widths = [stats_cc[i, cv2.CC_STAT_WIDTH] for i in valid_indices]
    comp_heights = [stats_cc[i, cv2.CC_STAT_HEIGHT] for i in valid_indices]
    comp_areas = [stats_cc[i, cv2.CC_STAT_AREA] for i in valid_indices]
    comp_centroids = [centroids[i] for i in valid_indices]
    
    # Urutkan komponen berdasarkan posisi X (kiri ke kanan)
    sorted_by_x = sorted(zip(comp_centroids, comp_widths, comp_heights), key=lambda item: item[0][0])
    sorted_centroids = [item[0] for item in sorted_by_x]
    
    # =========================================================================
    # A. FITUR GARIS DASAR (BASELINE STABILITY)
    # =========================================================================
    y_coords = [c[1] for c in sorted_centroids]
    baseline_dev_std = float(np.std(y_coords)) if len(y_coords) > 1 else 0.0
    
    # Waviness: selisih vertikal berturutan antar centroid
    y_diffs = np.abs(np.diff(y_coords)) if len(y_coords) > 1 else [0.0]
    baseline_waviness = float(np.mean(y_diffs))
    
    # =========================================================================
    # B. FITUR VARIASI UKURAN HURUF (LETTER SIZE & VARIATION)
    # =========================================================================
    h_mean = float(np.mean(comp_heights)) if comp_heights else 0.0
    h_std = float(np.std(comp_heights)) if len(comp_heights) > 1 else 0.0
    height_cov = (h_std / h_mean) if h_mean > 0 else 0.0
    
    w_mean = float(np.mean(comp_widths)) if comp_widths else 0.0
    w_std = float(np.std(comp_widths)) if len(comp_widths) > 1 else 0.0
    width_cov = (w_std / w_mean) if w_mean > 0 else 0.0
    
    char_aspect_ratios = [w / h if h > 0 else 0 for w, h in zip(comp_widths, comp_heights)]
    aspect_ratio_char_mean = float(np.mean(char_aspect_ratios)) if char_aspect_ratios else 0.0
    aspect_ratio_char_std = float(np.std(char_aspect_ratios)) if len(char_aspect_ratios) > 1 else 0.0
    
    # =========================================================================
    # C. FITUR KEMIRINGAN (SLANT ANGLE)
    # =========================================================================
    # Hitung gradien sobel untuk mendeteksi orientasi goresan vertikal
    sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    magnitude = np.sqrt(sobelx**2 + sobely**2)
    
    # Ambil orientasi piksel tepi yang signifikan
    edge_mask = (magnitude > np.percentile(magnitude, 75)) & (binary > 0)
    if np.sum(edge_mask) > 10:
        angles_rad = np.arctan2(sobely[edge_mask], sobelx[edge_mask])
        angles_deg = np.rad2deg(angles_rad)
        # Normalisasi ke rentang kemiringan vertikal [-90, 90]
        angles_deg = ((angles_deg + 90) % 180) - 90
        slant_angle_mean = float(np.mean(np.abs(angles_deg)))
        slant_angle_std = float(np.std(angles_deg))
    else:
        slant_angle_mean = 0.0
        slant_angle_std = 0.0
        
    # =========================================================================
    # D. FITUR SPASI & KERAPATAN TINTA (SPACING & INK DENSITY)
    # =========================================================================
    total_pixels = h_img * w_img
    ink_pixels = int(np.sum(binary == 255))
    ink_density = float(ink_pixels / total_pixels) if total_pixels > 0 else 0.0
    
    # Spasi antar komponen horizontal
    spacings = []
    for k in range(len(sorted_by_x) - 1):
        x_curr = sorted_by_x[k][0][0] + (sorted_by_x[k][1] / 2.0)
        x_next = sorted_by_x[k+1][0][0] - (sorted_by_x[k+1][1] / 2.0)
        space = max(0, x_next - x_curr)
        spacings.append(space)
        
    inter_comp_spacing_mean = float(np.mean(spacings)) if spacings else 0.0
    inter_comp_spacing_std = float(np.std(spacings)) if len(spacings) > 1 else 0.0
    num_components_density = float(len(valid_indices) / (w_img / 1000.0)) if w_img > 0 else 0.0
    
    # =========================================================================
    # E. FITUR KEKASARAN GORESAN (STROKE ROUGHNESS & THICKNESS)
    # =========================================================================
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    roughness_vals = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area >= min_area:
            perimeter = cv2.arcLength(cnt, True)
            # Rasio Isoperimetrik (P^2 / (4 * pi * A)). Nilai lingkaran sempurna = 1.0, bentuk kasar/bercabang >> 1.0
            r = (perimeter ** 2) / (4 * math.pi * area) if area > 0 else 0.0
            roughness_vals.append(r)
            
    stroke_roughness_mean = float(np.mean(roughness_vals)) if roughness_vals else 1.0
    stroke_roughness_std = float(np.std(roughness_vals)) if len(roughness_vals) > 1 else 0.0
    
    # Ketebalan goresan via Distance Transform
    dist_map = distance_transform_edt(binary == 255)
    ink_distances = dist_map[binary == 255]
    if len(ink_distances) > 0:
        stroke_thickness_mean = float(np.mean(ink_distances) * 2.0)
        stroke_thickness_std = float(np.std(ink_distances) * 2.0)
    else:
        stroke_thickness_mean = 0.0
        stroke_thickness_std = 0.0
        
    # =========================================================================
    # F. FITUR TEKSTUR GLCM (GRAY-LEVEL CO-OCCURRENCE MATRIX)
    # =========================================================================
    # Resample citra grayscale ke ukuran standar 64x256 untuk komputasi GLCM yang seragam
    gray_resized = cv2.resize(gray, (256, 64), interpolation=cv2.INTER_AREA)
    # Kuantisasi level ke 16 level abu-abu untuk mempercepat dan menstabilkan GLCM
    gray_quantized = (gray_resized // 16).astype(np.uint8)
    
    glcm = graycomatrix(gray_quantized, distances=[1, 3], angles=[0, np.pi/4, np.pi/2], levels=16, symmetric=True, normed=True)
    glcm_contrast = float(np.mean(graycoprops(glcm, 'contrast')))
    glcm_dissimilarity = float(np.mean(graycoprops(glcm, 'dissimilarity')))
    glcm_homogeneity = float(np.mean(graycoprops(glcm, 'homogeneity')))
    glcm_energy = float(np.mean(graycoprops(glcm, 'energy')))
    glcm_correlation = float(np.mean(graycoprops(glcm, 'correlation')))
    
    return {
        'baseline_dev_std': round(baseline_dev_std, 4),
        'baseline_waviness': round(baseline_waviness, 4),
        'height_mean': round(h_mean, 4),
        'height_std': round(h_std, 4),
        'height_cov': round(height_cov, 4),
        'width_mean': round(w_mean, 4),
        'width_std': round(w_std, 4),
        'width_cov': round(width_cov, 4),
        'aspect_ratio_char_mean': round(aspect_ratio_char_mean, 4),
        'aspect_ratio_char_std': round(aspect_ratio_char_std, 4),
        'slant_angle_mean': round(slant_angle_mean, 4),
        'slant_angle_std': round(slant_angle_std, 4),
        'ink_density': round(ink_density, 4),
        'inter_comp_spacing_mean': round(inter_comp_spacing_mean, 4),
        'inter_comp_spacing_std': round(inter_comp_spacing_std, 4),
        'num_components_density': round(num_components_density, 4),
        'stroke_roughness_mean': round(stroke_roughness_mean, 4),
        'stroke_roughness_std': round(stroke_roughness_std, 4),
        'stroke_thickness_mean': round(stroke_thickness_mean, 4),
        'stroke_thickness_std': round(stroke_thickness_std, 4),
        'glcm_contrast': round(glcm_contrast, 4),
        'glcm_dissimilarity': round(glcm_dissimilarity, 4),
        'glcm_homogeneity': round(glcm_homogeneity, 4),
        'glcm_energy': round(glcm_energy, 4),
        'glcm_correlation': round(glcm_correlation, 4)
    }

print("[INFO] Engine ekstraksi fitur disgrafia siap digunakan.")


---
## 6. Ekstraksi Fitur Seluruh Dataset ke DataFrame
Menjalankan ekstraksi fitur pada seluruh gambar di dataset dan menggabungkannya ke dalam sebuah DataFrame tabular lengkap.


In [ ]:
def build_features_dataset(base_dir):
    """
    Mengekstrak seluruh fitur dari dataset citra dan mengembalikan DataFrame.
    """
    base_path = Path(base_dir).resolve()
    classes = [d for d in base_path.iterdir() if d.is_dir() and not d.name.startswith('.')]
    
    rows = []
    print("[INFO] Memulai proses ekstraksi fitur dataset...")
    
    for cls_dir in classes:
        cls_name = cls_dir.name
        img_files = list(cls_dir.glob("*.jpg")) + list(cls_dir.glob("*.png")) + list(cls_dir.glob("*.jpeg"))
        print(f" - Memproses kelas '{cls_name}': {len(img_files)} file gambar...")
        
        for fpath in img_files:
            try:
                features = extract_handwriting_features(fpath)
                row = {
                    'filename': fpath.name,
                    'label': cls_name,
                    'target': 1 if 'potential dysgraphia' == cls_name.lower() or cls_name == 'Potential Dysgraphia' else 0
                }
                row.update(features)
                rows.append(row)
            except Exception as e:
                print(f"[ERROR] Gagal memproses {fpath.name}: {e}")
                
    df = pd.DataFrame(rows)
    print(f"[SUCCESS] Selesai mengekstrak {len(df)} gambar dengan {df.shape[1] - 3} fitur per gambar.")
    return df

df_features = build_features_dataset(DATASET_DIR)
print("\nPreview 5 Baris Pertama Dataset Fitur:")
display(df_features.head())


---
## 7. Analisis Statistik & Uji Signifikansi Fitur (Mann-Whitney U Test)
Untuk mengetahui fitur mana yang **memiliki perbedaan nyata secara statistik** antara kelas `Low Potential Dysgraphia` dan `Potential Dysgraphia`, kita melakukan uji hipotesis non-parametrik **Mann-Whitney U Test** (karena distribusi fitur citra tidak selalu normal).

* **Nilai p-value < 0.05**: Menunjukkan bahwa fitur tersebut berbeda secara signifikan antara kedua kelas (*statistically significant*).
* **Nilai p-value < 0.001**: Fitur sangat diskriminatif (*highly significant*).


In [ ]:
def compute_feature_significance(df):
    """
    Menghitung mean per kelas dan nilai p-value (Mann-Whitney U test) untuk setiap fitur.
    """
    feature_cols = [c for c in df.columns if c not in ['filename', 'label', 'target']]
    
    low_pot = df[df['label'] == 'Low Potential Dysgraphia']
    pot = df[df['label'] == 'Potential Dysgraphia']
    
    results = []
    for col in feature_cols:
        val_low = low_pot[col].dropna()
        val_pot = pot[col].dropna()
        
        # Mann-Whitney U test
        stat_u, p_val = stats.mannwhitneyu(val_low, val_pot, alternative='two-sided')
        
        mean_low = val_low.mean()
        mean_pot = val_pot.mean()
        diff_pct = ((mean_pot - mean_low) / mean_low * 100) if mean_low != 0 else 0.0
        
        results.append({
            'Fitur': col,
            'Mean (Low Potential)': round(mean_low, 4),
            'Mean (Potential)': round(mean_pot, 4),
            'Selisih (%)': round(diff_pct, 2),
            'p-value': p_val,
            'Signifikan (p < 0.05)': 'Ya (***)' if p_val < 0.001 else ('Ya (*)' if p_val < 0.05 else 'Tidak')
        })
        
    res_df = pd.DataFrame(results).sort_values(by='p-value')
    return res_df

stat_significance_df = compute_feature_significance(df_features)
print("Peringkat Fitur Berdasarkan Nilai Signifikansi (p-value):")
display(stat_significance_df)


---
## 8. Visualisasi Fitur Paling Diskriminatif (Top Features Boxplot)
Menampilkan boxplot untuk 6 fitur dengan nilai signifikansi tertinggi guna mengamati perbandingan sebaran nilai antar kedua kelas secara visual.


In [ ]:
top_features = stat_significance_df['Fitur'].head(6).tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
palette = ['#2b5c8f', '#d95f02']

for idx, feat in enumerate(top_features):
    pval = stat_significance_df.loc[stat_significance_df['Fitur'] == feat, 'p-value'].values[0]
    
    sns.boxplot(
        x='label', y=feat, data=df_features, ax=axes[idx],
        hue='label', palette=palette, legend=False,
        boxprops=dict(alpha=0.8)
    )
    # Tambahkan titik data individual (strip plot)
    sns.stripplot(x='label', y=feat, data=df_features, ax=axes[idx], color='black', alpha=0.25, jitter=0.2, size=4)
    
    axes[idx].set_title(f"{feat}\n(p-value = {pval:.2e})", fontsize=11, fontweight='bold')
    axes[idx].set_xlabel("Kelas", fontweight='bold')
    axes[idx].set_ylabel("Nilai Fitur", fontweight='bold')
    axes[idx].grid(True, linestyle='--', alpha=0.6)

plt.suptitle("Top 6 Fitur Tulisan Tangan Paling Signifikan Membedakan Disgrafia", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


---
## 9. Analisis Korelasi Antar Fitur (Correlation Matrix)
Memeriksa matriks korelasi Pearson antar fitur untuk mengidentifikasi apakah terdapat pasangan fitur yang mengalami multikolinearitas tinggi ($r > 0.85$).


In [ ]:
feature_cols = [c for c in df_features.columns if c not in ['filename', 'label', 'target']]
corr_matrix = df_features[feature_cols].corr()

plt.figure(figsize=(15, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, cmap='vlag', vmin=-1, vmax=1, center=0,
    square=True, linewidths=.5, cbar_kws={"shrink": .8},
    annot=False
)
plt.title("Heatmap Matriks Korelasi Antar Fitur Tulisan Tangan", fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

# Deteksi pasangan fitur dengan korelasi sangat tinggi (> 0.85)
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        col1 = corr_matrix.columns[i]
        col2 = corr_matrix.columns[j]
        val = corr_matrix.iloc[i, j]
        if abs(val) > 0.85:
            high_corr_pairs.append((col1, col2, round(val, 3)))
            
print(f"Pasangan fitur dengan korelasi tinggi (|r| > 0.85) berjumlah {len(high_corr_pairs)}:")
for c1, c2, r_val in high_corr_pairs:
    print(f" - {c1:<25} <--> {c2:<25} (r = {r_val})")


---
## 10. Reduksi Dimensi & Visualisasi Ruang Fitur (PCA & t-SNE 2D)
Memetakan seluruh vektor fitur (25 dimensi) ke ruang 2D menggunakan:
1. **PCA (*Principal Component Analysis*)**: Menangkap variansi linear global terbesar.
2. **t-SNE (*t-Distributed Stochastic Neighbor Embedding*)**: Menangkap struktur manifold non-linear lokal.

Visualisasi ini membantu melihat apakah sampel `Low Potential Dysgraphia` dan `Potential Dysgraphia` membentuk klaster terpisah dalam ruang fitur.


In [ ]:
# Standarisasi fitur (mean = 0, std = 1)
X = df_features[feature_cols].values
y = df_features['label'].values
y_target = df_features['target'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 1. PCA 2D
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
var_pca = pca.explained_variance_ratio_ * 100

# 2. t-SNE 2D
tsne = TSNE(n_components=2, random_state=RANDOM_STATE, perplexity=30)
X_tsne = tsne.fit_transform(X_scaled)

# Visualisasi PCA & t-SNE
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
colors = {'Low Potential Dysgraphia': '#2b5c8f', 'Potential Dysgraphia': '#d95f02'}

# Scatter Plot PCA
for label_name in np.unique(y):
    mask = (y == label_name)
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], c=colors[label_name], label=label_name, alpha=0.8, edgecolors='black', s=50)
axes[0].set_title(f'PCA 2D Projection (Total Var: {var_pca[0]+var_pca[1]:.1f}%)', fontsize=12, fontweight='bold')
axes[0].set_xlabel(f'Principal Component 1 ({var_pca[0]:.1f}%)', fontweight='bold')
axes[0].set_ylabel(f'Principal Component 2 ({var_pca[1]:.1f}%)', fontweight='bold')
axes[0].legend(frameon=True)
axes[0].grid(True, linestyle='--', alpha=0.6)

# Scatter Plot t-SNE
for label_name in np.unique(y):
    mask = (y == label_name)
    axes[1].scatter(X_tsne[mask, 0], X_tsne[mask, 1], c=colors[label_name], label=label_name, alpha=0.8, edgecolors='black', s=50)
axes[1].set_title('t-SNE 2D Manifold Projection', fontsize=12, fontweight='bold')
axes[1].set_xlabel('t-SNE Dimension 1', fontweight='bold')
axes[1].set_ylabel('t-SNE Dimension 2', fontweight='bold')
axes[1].legend(frameon=True)
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.suptitle("Visualisasi Pemisahan Ruang Fitur Citra Tulisan Tangan", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


---
## 11. Peringkat Kepentingan Fitur (*Random Forest Feature Importance*)
Menggunakan model *Random Forest* (100 trees) untuk mengevaluasi *Gini Importance / Mean Decrease in Impurity (MDI)* dari tiap fitur dalam mengklasifikasikan disgrafia.


In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, max_depth=6)
rf.fit(X_scaled, y_target)

importances = rf.feature_importances_
feat_imp_df = pd.DataFrame({
    'Fitur': feature_cols,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(
    x='Importance', y='Fitur', data=feat_imp_df,
    hue='Fitur', palette='viridis', edgecolor='black', legend=False
)
plt.title("Peringkat Kepentingan Fitur (Random Forest Feature Importance)", fontsize=13, fontweight='bold', pad=15)
plt.xlabel("Gini Importance Score", fontweight='bold')
plt.ylabel("Nama Fitur", fontweight='bold')
plt.tight_layout()
plt.show()

print("Top 10 Fitur Paling Berpengaruh:")
display(feat_imp_df.head(10))


---
## 12. Ekspor Dataset Fitur ke CSV
Menyimpan dataset fitur yang telah diekstrak ke dalam file tabular **`dysgraphia_features.csv`**.

File ini siap digunakan langsung pada tahap pemodelan *Machine Learning* (SVM, Random Forest, XGBoost, CatBoost, dll.) tanpa perlu mengekstrak ulang dari citra asli.


In [ ]:
csv_output_path = "dysgraphia_features.csv"
df_features.to_csv(csv_output_path, index=False)
print(f"[SUCCESS] Dataset fitur berhasil disimpan ke file: '{csv_output_path}'")
print(f"Total baris: {df_features.shape[0]}, Total kolom: {df_features.shape[1]}")
